In [ ]:
!pip install -q openai datasets numpy matplotlib
!pip install -q --no-deps "tau-bench @ git+https://github.com/sierra-research/tau-bench"

In [ ]:
import difflib
import json
import math
import os
import random
import re
import subprocess
import sys
import tempfile
import time
import types
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset
from openai import OpenAI
from openai.types.chat import ChatCompletionMessage

# tau-bench's package init imports litellm for its own user simulator; this notebook
# builds its own simulator, so a stub stands in and keeps openai unpinned
sys.modules.setdefault("litellm", types.SimpleNamespace(completion=None))
from tau_bench.envs.retail.data import load_data
from tau_bench.envs.retail.tasks_test import TASKS_TEST
from tau_bench.envs.retail.tools import ALL_TOOLS
from tau_bench.envs.retail.wiki import WIKI

os.environ["OPENAI_API_KEY"] = ""

client = OpenAI()
MODEL = "gpt-5-mini"
AGENT_MODELS = ["gpt-5-mini", "gpt-5-nano"]
USER_MODEL = "gpt-4.1-mini"
SEED = 0

random.seed(SEED)

# Agent Evaluation

---
# 1) Introduction

Evaluating an agent is not evaluating a model on a static prompt. An agent run is a
**trajectory**: many model calls, tool executions and environment state changes, ending in an
outcome that must be checked by something other than string matching. Four things make it hard:

| Problem | Why static-benchmark habits fail | What is built here |
|---|---|---|
| Stochastic outcomes | One run per task hides how often the agent fails on a task it *can* solve | pass@k vs **pass^k** estimators with confidence intervals |
| Outcome checking | The agent's final message is not the result; the repository or database is | Hidden-test grading (SWE-bench / Terminal-Bench style) and database-state grading (τ-bench style) |
| Process quality | Two successful runs can differ in cost, policy violations and wasted steps | Trajectory grading by a judge model, checked against ground truth |
| Contamination | Public benchmarks leak into pretraining data | Perturbation and verbatim-completion probes |

The two harnesses use real benchmarks: **HumanEval+** tasks solved by an agent with a shell inside
a sandbox and graded by hidden tests, and the **τ-bench retail** domain (its real database, tools,
policy and test tasks), driven by a simulated user and graded by the final database state.

---
# 2) Setup

Two agents are compared: `gpt-5-mini` and `gpt-5-nano`, both at `reasoning_effort="minimal"` so
that failures are frequent enough to measure. The simulated user in the τ-bench harness is
`gpt-4.1-mini`. Every call's token usage is recorded.

In [ ]:
USAGE: dict[str, dict[str, int]] = {}


def call(
    model: str,
    messages: list[dict],
    tools: list[dict] | None = None,
    effort: str | None = "minimal",
) -> ChatCompletionMessage:
    """One chat completion; records token usage per model."""
    kwargs = {"tools": tools} if tools else {}
    if model.startswith("gpt-5"):
        kwargs["reasoning_effort"] = effort
    for attempt in range(4):
        try:
            response = client.chat.completions.create(
                model=model, messages=messages, **kwargs
            )
            break
        except Exception:
            if attempt == 3:
                raise
            time.sleep(2**attempt)
    usage = USAGE.setdefault(model, {"input": 0, "output": 0, "calls": 0})
    usage["input"] += response.usage.prompt_tokens
    usage["output"] += response.usage.completion_tokens
    usage["calls"] += 1
    return response.choices[0].message


print(call(MODEL, [{"role": "user", "content": "Reply with the word ready."}]).content)

---
# 3) The math

For a task with $n$ recorded trials of which $c$ succeeded, two different questions can be asked.

**pass@k** — will *at least one* of $k$ attempts succeed? (Chen et al., 2021)
$$
\widehat{\text{pass@}k} = 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}} .
$$

**pass^k** — will *all* $k$ attempts succeed? (Yao et al., τ-bench, 2024)
$$
\widehat{\text{pass}^k} = \frac{\binom{c}{k}}{\binom{n}{k}} .
$$

Both are unbiased estimators of the probability over a random draw of $k$ of the $n$ trials, and
both are averaged over tasks. pass@k is the right metric when a verifier can pick the successful
attempt (best-of-k with tests). pass^k is the right metric for a deployed agent that answers a
user once, many times: a customer-service agent with pass@1 = 0.6 and pass^8 = 0.2 fails most
customers who come back eight times.

If a task's per-trial success probability is $p$, then pass@k $= 1-(1-p)^k$ rises with $k$ and
pass^k $= p^k$ falls with $k$; the gap between the two curves is a direct measure of
**inconsistency**.

**Uncertainty.** Benchmarks with tens of tasks have wide intervals. The interval used here is the
percentile bootstrap over tasks (resample tasks with replacement, recompute the mean metric).

---
# 4) From-scratch implementation

## 4.1 Estimators

In [ ]:
def pass_at_k(n: int, c: int, k: int) -> float:
    """Unbiased P(at least one of k sampled trials succeeds)."""
    if n - c < k:
        return 1.0
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)


def pass_hat_k(n: int, c: int, k: int) -> float:
    """Unbiased P(all k sampled trials succeed)."""
    return math.comb(c, k) / math.comb(n, k)


def bootstrap_ci(
    values: list[float], num: int = 2000, alpha: float = 0.05
) -> tuple[float, float]:
    """Percentile bootstrap interval of the mean over tasks."""
    rng = np.random.default_rng(SEED)
    arr = np.asarray(values)
    means = rng.choice(arr, size=(num, len(arr)), replace=True).mean(axis=1)
    return float(np.quantile(means, alpha / 2)), float(
        np.quantile(means, 1 - alpha / 2)
    )


# check against brute-force enumeration of every k-subset of trials
outcomes = [1, 0, 1, 1, 0]
n, c = len(outcomes), sum(outcomes)
for k in range(1, n + 1):
    subsets = list(combinations(outcomes, k))
    brute_any = sum(any(s) for s in subsets) / len(subsets)
    brute_all = sum(all(s) for s in subsets) / len(subsets)
    print(
        f"k={k}: pass@k {pass_at_k(n, c, k):.3f} (brute {brute_any:.3f}), "
        f"pass^k {pass_hat_k(n, c, k):.3f} (brute {brute_all:.3f})"
    )

## 4.2 A sandboxed code-agent harness

The structure follows Terminal-Bench and SWE-bench: the agent gets a task description and a
**workspace**, acts only through tools, and is graded afterwards by tests it never saw.

```
task --> agent loop --(bash / write_file)--> sandboxed workspace
                     \--(submit)--> hidden tests run in the workspace --> pass / fail
```

### 4.2.1 The sandbox

Model-written code runs on this machine, so it is isolated. On macOS the harness uses
`sandbox-exec` (Seatbelt, the mechanism Codex CLI uses): no network, and file writes only inside
the workspace (temporary files included). On Linux the equivalent is Landlock + seccomp, or a container; the fallback below
is a plain subprocess with a timeout, which is **not** isolation.

In [ ]:
def run_sandboxed(command: str, workspace: str, timeout: int = 20) -> str:
    """Run a shell command in the workspace; returns combined output (truncated)."""
    if sys.platform == "darwin":
        root = os.path.realpath(workspace)
        profile = (
            "(version 1)(allow default)(deny network*)(deny file-write*)"
            f'(allow file-write* (subpath "{root}") (literal "/dev/null"))'
        )
        argv = ["sandbox-exec", "-p", profile, "/bin/bash", "-c", command]
    else:
        argv = ["/bin/bash", "-c", command]
    try:
        # temporary files go inside the workspace too, so parallel trials cannot touch each other
        env = {**os.environ, "TMPDIR": workspace}
        result = subprocess.run(
            argv,
            cwd=workspace,
            env=env,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        output = result.stdout + result.stderr
    except subprocess.TimeoutExpired:
        output = f"Error: command timed out after {timeout}s"
    return output[-4000:] or "(no output)"


with tempfile.TemporaryDirectory() as ws:
    print(run_sandboxed("echo hi > inside.txt && cat inside.txt", ws))
    print(run_sandboxed("echo x > ~/escape.txt", ws).strip())
    print(
        run_sandboxed(
            f"{sys.executable} -c \"import urllib.request; urllib.request.urlopen('http://example.com', timeout=3)\" 2>&1 | tail -1",
            ws,
        ).strip()
    )

### 4.2.2 Tasks, tools and the agent loop

Each HumanEval+ task becomes a workspace containing `solution.py` with the function signature
and docstring. The agent must implement it, may run its own checks with `bash`, and ends with
`submit`. The hidden HumanEval+ tests (on average about 750 inputs per task, many more than the
original HumanEval) are copied in only after submission.

In [ ]:
humaneval = load_dataset("evalplus/humanevalplus", split="test")
CODE_TASKS = sorted(random.Random(SEED).sample(range(len(humaneval)), 40))
NUM_TRIALS = 4
MAX_STEPS = 12

CODE_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "bash",
            "description": "Run a bash command in the workspace and return its output.",
            "parameters": {
                "type": "object",
                "properties": {"command": {"type": "string"}},
                "required": ["command"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Overwrite a file in the workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string"},
                    "content": {"type": "string"},
                },
                "required": ["path", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "submit",
            "description": "Finish the task. Call when solution.py is complete.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
]
CODE_SYSTEM = (
    "You are a coding agent working in a sandboxed directory. Implement the function in "
    "solution.py so that it satisfies its docstring, including edge cases. Use the tools; test "
    "your code with python3 before submitting. Call submit when done."
)


def code_agent(model: str, task: dict, workspace: str) -> dict:
    """Run the agent on one task; returns the trajectory and step count."""
    with open(os.path.join(workspace, "solution.py"), "w") as f:
        f.write(task["prompt"])
    messages = [
        {"role": "system", "content": CODE_SYSTEM},
        {
            "role": "user",
            "content": f"Implement `{task['entry_point']}` in solution.py:\n\n{task['prompt']}",
        },
    ]
    for step in range(MAX_STEPS):
        message = call(model, messages, CODE_TOOLS)
        messages.append(message.model_dump(exclude_none=True))
        if not message.tool_calls:
            messages.append(
                {"role": "user", "content": "Use the tools, and call submit when done."}
            )
            continue
        for tool_call in message.tool_calls:
            args = json.loads(tool_call.function.arguments or "{}")
            name = tool_call.function.name
            if name == "submit":
                return {"messages": messages, "steps": step + 1, "submitted": True}
            if name == "write_file":
                path = os.path.join(
                    workspace, os.path.basename(args.get("path", "solution.py"))
                )
                with open(path, "w") as f:
                    f.write(args.get("content", ""))
                result = f"wrote {os.path.basename(path)}"
            else:
                result = run_sandboxed(args.get("command", ""), workspace)
            messages.append(
                {"role": "tool", "tool_call_id": tool_call.id, "content": result}
            )
    return {"messages": messages, "steps": MAX_STEPS, "submitted": False}

### 4.2.3 Hidden-test grading

The grader appends the HumanEval+ `check` function to the agent's final `solution.py` and runs it
in the same sandbox. Pass means every hidden input matched the reference output.

In [ ]:
def grade_code(task: dict, workspace: str) -> bool:
    """Run the hidden tests against the agent's solution.py."""
    test_path = os.path.join(workspace, "hidden_test.py")
    with open(test_path, "w") as f:
        f.write(
            "from solution import *\n"
            + task["test"]
            + f"\ncheck({task['entry_point']})\nprint('ALL TESTS PASSED')\n"
        )
    output = run_sandboxed(f"{sys.executable} hidden_test.py", workspace, timeout=60)
    return "ALL TESTS PASSED" in output


def run_code_trial(model: str, task_index: int, trial: int) -> dict:
    """One full trial: fresh workspace, agent run, hidden-test grade."""
    task = humaneval[task_index]
    with tempfile.TemporaryDirectory() as workspace:
        trajectory = code_agent(model, task, workspace)
        passed = grade_code(task, workspace)
    return {
        "model": model,
        "task": task_index,
        "trial": trial,
        "passed": passed,
        **trajectory,
    }


canonical = humaneval[CODE_TASKS[0]]
with tempfile.TemporaryDirectory() as ws:
    with open(os.path.join(ws, "solution.py"), "w") as f:
        f.write(canonical["prompt"] + canonical["canonical_solution"])
    print(
        f"Canonical solution of {canonical['task_id']} passes hidden tests: {grade_code(canonical, ws)}"
    )
    with open(os.path.join(ws, "solution.py"), "w") as f:
        f.write(canonical["prompt"] + "    return None\n")
    print(f"Stub solution passes hidden tests: {grade_code(canonical, ws)}")

### 4.2.4 Running the benchmark

In [ ]:
def run_grid(
    trial_fn, models: list[str], tasks: list[int], trials: int, workers: int = 16
) -> list[dict]:
    """Every (model, task, trial) in parallel."""
    jobs = [(m, t, k) for m in models for t in tasks for k in range(trials)]
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(pool.map(lambda job: trial_fn(*job), jobs))


def group_by_task(records: list[dict], model: str, key: str) -> dict[int, list[bool]]:
    """Per-task list of trial outcomes for one model."""
    by_task: dict[int, list[bool]] = {}
    for r in records:
        if r["model"] == model:
            by_task.setdefault(r["task"], []).append(bool(r[key]))
    return by_task


def summarize(records: list[dict], key: str, trials: int) -> None:
    """pass@k and pass^k per model with bootstrap intervals over tasks."""
    for model in sorted({r["model"] for r in records}):
        by_task = group_by_task(records, model, key)
        row = []
        for k in range(1, trials + 1):
            at = [pass_at_k(len(v), sum(v), k) for v in by_task.values()]
            hat = [pass_hat_k(len(v), sum(v), k) for v in by_task.values()]
            lo, hi = bootstrap_ci(hat)
            row.append(
                f"k={k}: @{np.mean(at):.2f} ^{np.mean(hat):.2f} [{lo:.2f},{hi:.2f}]"
            )
        steps = np.mean([r["steps"] for r in records if r["model"] == model])
        print(f"{model:<11} " + " | ".join(row) + f" | mean steps {steps:.1f}")


start = time.time()
code_records = run_grid(run_code_trial, AGENT_MODELS, CODE_TASKS, NUM_TRIALS)
print(f"{len(code_records)} trials in {time.time() - start:.0f}s")
summarize(code_records, "passed", NUM_TRIALS)

In [ ]:
def plot_pass_curves(records: list[dict], key: str, trials: int, title: str) -> None:
    """pass@k rising and pass^k falling with k, per model."""
    plt.figure(figsize=(6, 3.5))
    for model in sorted({r["model"] for r in records}):
        by_task = group_by_task(records, model, key)
        ks = range(1, trials + 1)
        at = [
            np.mean([pass_at_k(len(v), sum(v), k) for v in by_task.values()])
            for k in ks
        ]
        hat = [
            np.mean([pass_hat_k(len(v), sum(v), k) for v in by_task.values()])
            for k in ks
        ]
        line = plt.plot(ks, at, marker="o", label=f"{model} pass@k")[0]
        plt.plot(
            ks,
            hat,
            marker="s",
            ls="--",
            color=line.get_color(),
            label=f"{model} pass^k",
        )
    plt.xlabel("k")
    plt.ylabel("rate")
    plt.title(title)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


plot_pass_curves(code_records, "passed", NUM_TRIALS, "HumanEval+ agent")

`gpt-5-mini` loses little when every trial must succeed (pass^k), while `gpt-5-nano` drops
sharply even though its pass@k is high. The smaller model *can* solve almost every task but does so unreliably; pass@1 alone hides that, and
the bootstrap intervals show how little a single run of a small benchmark
pins down.

## 4.3 A τ-bench-style conversational harness

τ-bench evaluates agents that must follow a **policy** while helping a **user** they cannot see
the goal of. The retail domain ships a database (500 users, 1,000 orders, 50 products), 16 tools
that read and mutate it, a 5,700-character policy, and test tasks, each a user instruction plus
the gold sequence of write actions.

```
user simulator (knows the instruction) <-> agent (knows the policy, has tools) -> database
                                                                                  |
                     reward = 1 if final DB == DB after gold actions, and required info was said
```

Grading by **database state** is what makes this robust: it does not matter how the agent
phrased things or in which order it looked things up, only whether the world ended in the
right state.

In [ ]:
TAU_TOOLS = {tool.get_info()["function"]["name"]: tool for tool in ALL_TOOLS}
TAU_SCHEMAS = [tool.get_info() for tool in ALL_TOOLS]
TAU_TASKS = list(range(20))
MAX_TURNS = 30
print(f"{len(TAU_TOOLS)} tools: {sorted(TAU_TOOLS)}")
print(f"Task 0 instruction: {TASKS_TEST[0].instruction[:200]}...")
print(f"Gold actions: {[a.name for a in TASKS_TEST[0].actions]}")

### 4.3.1 Database-state reward

Replaying the gold actions on a fresh copy of the database gives the target state. The agent's
final state is compared by a canonical hash (dicts sorted by key), exactly as τ-bench does. A task
may also list `outputs`, strings the agent must have told the user.

In [ ]:
def canonical(obj: object) -> object:
    """Order-independent hashable view of nested dicts/lists."""
    if isinstance(obj, dict):
        return tuple((k, canonical(v)) for k, v in sorted(obj.items()))
    if isinstance(obj, list):
        return tuple(canonical(v) for v in obj)
    return obj


def apply_action(data: dict, name: str, kwargs: dict) -> str:
    """Execute one tool against the database (mutates it)."""
    try:
        return TAU_TOOLS[name].invoke(data, **kwargs)
    except Exception as error:
        return f"Error: {error}"


def tau_reward(task_index: int, final_data: dict, agent_text: str) -> dict:
    """1 if final DB matches the gold replay and required outputs were stated."""
    task = TASKS_TEST[task_index]
    gold = load_data()
    for action in task.actions:
        apply_action(gold, action.name, action.kwargs)
    state_ok = hash(canonical(final_data)) == hash(canonical(gold))
    outputs_ok = all(
        o.lower().replace(",", "") in agent_text.lower().replace(",", "")
        for o in task.outputs
    )
    return {
        "state_ok": state_ok,
        "outputs_ok": outputs_ok,
        "reward": state_ok and outputs_ok,
    }


untouched = load_data()
print(f"Doing nothing on task 0: {tau_reward(0, untouched, '')}")
replayed = load_data()
for a in TASKS_TEST[0].actions:
    apply_action(replayed, a.name, a.kwargs)
print(f"Replaying gold actions: {tau_reward(0, replayed, '')}")

### 4.3.2 The user simulator and the agent loop

The simulated user gets the task instruction and is told to reveal information only when asked,
one piece at a time, and to end with `###STOP###` when the goal is met or cannot be met. The
agent gets the policy as its system prompt and the tool schemas. A turn alternates: agent
(possibly several tool calls) then user.

In [ ]:
USER_SYSTEM = (
    "You are a customer talking to a retail customer-service agent. Your goal:\n{instruction}\n\n"
    "Rules: write only your next message as the customer. Reveal information only when the agent "
    "asks for it. Do not invent information not in your goal. When your goal is fully handled, or "
    "cannot be handled, reply with exactly ###STOP###."
)


def tau_trial(model: str, task_index: int, trial: int) -> dict:
    """One simulated conversation; returns reward, trajectory and step statistics."""
    task = TASKS_TEST[task_index]
    data = load_data()
    agent_messages = [{"role": "system", "content": WIKI}]
    user_messages = [
        {"role": "system", "content": USER_SYSTEM.format(instruction=task.instruction)},
        {"role": "user", "content": "Hi! How can I help you today?"},
    ]
    said, tool_calls, tool_errors = [], 0, 0
    for turn in range(MAX_TURNS):
        user_text = call(USER_MODEL, user_messages, effort=None).content or ""
        if "###STOP###" in user_text:
            break
        user_messages.append({"role": "assistant", "content": user_text})
        agent_messages.append({"role": "user", "content": user_text})
        # at most 20 tool rounds per user turn
        for _ in range(20):
            message = call(model, agent_messages, TAU_SCHEMAS)
            agent_messages.append(message.model_dump(exclude_none=True))
            if not message.tool_calls:
                break
            for tc in message.tool_calls:
                result = apply_action(
                    data, tc.function.name, json.loads(tc.function.arguments or "{}")
                )
                tool_calls += 1
                tool_errors += result.startswith("Error")
                agent_messages.append(
                    {"role": "tool", "tool_call_id": tc.id, "content": result}
                )
        said.append(message.content or "")
        user_messages.append({"role": "user", "content": message.content or ""})
    reward = tau_reward(task_index, data, "\n".join(said))
    return {
        "model": model,
        "task": task_index,
        "trial": trial,
        "steps": turn + 1,
        "tool_calls": tool_calls,
        "tool_errors": tool_errors,
        "messages": agent_messages,
        **reward,
    }


example = tau_trial(MODEL, 0, 0)
print(
    {
        k: example[k]
        for k in [
            "reward",
            "state_ok",
            "outputs_ok",
            "steps",
            "tool_calls",
            "tool_errors",
        ]
    }
)
for m in example["messages"][1:7]:
    text = m.get("content") or json.dumps(
        [t["function"] for t in m.get("tool_calls", [])]
    )
    print(f"[{m['role']}] {str(text)[:160]}")

In [ ]:
start = time.time()
tau_records = run_grid(tau_trial, AGENT_MODELS, TAU_TASKS, NUM_TRIALS, workers=12)
print(f"{len(tau_records)} conversations in {time.time() - start:.0f}s")
summarize(tau_records, "reward", NUM_TRIALS)
plot_pass_curves(tau_records, "reward", NUM_TRIALS, "tau-bench retail")

Conversational tasks are much less consistent than code. For both models pass^k falls far below pass@1,
the same shape as the τ-bench paper's results. Failures come from the agent (wrong item variant, skipped confirmation, acting before
authentication) and from the user simulator (revealing a goal differently each time), which is
exactly the variance a deployed agent faces.

## 4.4 Trajectory grading

Outcome rewards say *whether* a run succeeded, not *how*. A judge model reads the full trajectory
with the policy and grades it against a rubric. Before a judge can replace ground truth it must
be **validated against it**: here its success verdict is compared with the database reward on the
same trajectories, and its policy-violation flags are reported per outcome.

In [ ]:
JUDGE_PROMPT = """You are auditing a customer-service agent transcript against its policy.

POLICY:
{policy}

TRANSCRIPT (agent's view, including tool calls and results):
{transcript}

Return JSON with keys:
- "success": true if the customer's request was correctly and completely handled
- "policy_violation": true if the agent broke any policy rule (e.g. acted without explicit confirmation, skipped authentication, made multiple modification calls where one was required)
- "wasted_steps": integer count of clearly unnecessary tool calls
- "reason": one sentence"""


def render(messages: list[dict]) -> str:
    """Compact transcript for the judge."""
    lines = []
    for m in messages[1:]:
        if m.get("tool_calls"):
            lines += [
                f"[agent tool call] {t['function']['name']}({t['function']['arguments']})"
                for t in m["tool_calls"]
            ]
        elif m["role"] == "tool":
            lines.append(f"[tool result] {m['content'][:300]}")
        else:
            lines.append(f"[{m['role']}] {m.get('content') or ''}")
    return "\n".join(lines)


def judge(record: dict) -> dict:
    """Rubric verdict from the judge model."""
    reply = call(
        MODEL,
        [
            {
                "role": "user",
                "content": JUDGE_PROMPT.format(
                    policy=WIKI, transcript=render(record["messages"])
                ),
            }
        ],
        effort="low",
    ).content
    match = re.search(r"\{.*\}", reply or "", re.S)
    return json.loads(match.group(0)) if match else {"success": None}


with ThreadPoolExecutor(max_workers=12) as pool:
    verdicts = list(pool.map(judge, tau_records))
truth = np.array([r["reward"] for r in tau_records])
said = np.array([bool(v.get("success")) for v in verdicts])
agree = (truth == said).mean()
p_yes = truth.mean() * said.mean() + (1 - truth.mean()) * (1 - said.mean())
kappa = (agree - p_yes) / (1 - p_yes)
print(f"Judge vs database reward: agreement {agree:.0%}, Cohen's kappa {kappa:.2f}")
print(
    f"  judge says success on failed runs:    {(said & ~truth).sum()} of {(~truth).sum()}"
)
print(
    f"  judge says failure on successful runs: {(~said & truth).sum()} of {truth.sum()}"
)
violations = np.array([bool(v.get("policy_violation")) for v in verdicts])
print(
    f"Policy violations flagged: {violations[truth].mean():.0%} of successful runs, "
    f"{violations[~truth].mean():.0%} of failed runs"
)

The judge agrees with the database reward on only part of the conversations (low Cohen's κ),
and most of its errors call **failed runs successful**: transcripts in which the agent
confidently announced a completed exchange of the wrong item read as successes. A judge reading
the conversation cannot see the database. Outcome checks catch this; judges are for qualities
that have no outcome check (tone, policy adherence), and only after validation like this.

## 4.5 Contamination

A model that saw a benchmark during pretraining can score well by recall. Two probes that need no
access to the training data:

- **Perturbation.** Rewrite each task so that its meaning and tests are unchanged but its surface
  form is new: a paraphrased docstring and a renamed function. Memorised solutions do not transfer;
  understood ones do. HumanEval tests call the function through `candidate`, so renaming does not
  touch them.
- **Verbatim completion.** Give the first half of a task's docstring and ask for the continuation.
  Exact reproduction of a long, specific text is evidence it was memorised. The control is the
  same probe on the paraphrased docstrings, which cannot have been seen.

In [ ]:
PARAPHRASE = (
    "Rewrite this Python function stub. Rename the function to a new descriptive name, rewrite the "
    "docstring in your own words (keep every requirement and keep the examples' values but with the "
    "new name), keep parameter names and types. Return only the code.\n\n{prompt}"
)


def perturb(task_index: int) -> dict:
    """A paraphrased, renamed copy of a HumanEval+ task with the same hidden tests."""
    task = dict(humaneval[task_index])
    code = call(
        MODEL,
        [{"role": "user", "content": PARAPHRASE.format(prompt=task["prompt"])}],
        effort="low",
    ).content
    code = re.sub(r"^```(?:python)?\n|```$", "", code.strip(), flags=re.M)
    names = re.findall(r"^def (\w+)\(", code, flags=re.M)
    helpers = re.findall(r"^def (\w+)\(", task["prompt"], flags=re.M)[:-1]
    new_name = (
        [n for n in names if n not in helpers][-1] if names else task["entry_point"]
    )
    return {**task, "prompt": code + "\n", "entry_point": new_name}


with ThreadPoolExecutor(max_workers=16) as pool:
    perturbed = dict(zip(CODE_TASKS, pool.map(perturb, CODE_TASKS)))
print(perturbed[CODE_TASKS[0]]["prompt"][:400])

In [ ]:
def run_perturbed_trial(model: str, task_index: int, trial: int) -> dict:
    """Same harness on the perturbed task."""
    task = perturbed[task_index]
    with tempfile.TemporaryDirectory() as workspace:
        trajectory = code_agent(model, task, workspace)
        passed = grade_code(task, workspace)
    return {
        "model": model,
        "task": task_index,
        "trial": trial,
        "passed": passed,
        **trajectory,
    }


perturbed_records = run_grid(run_perturbed_trial, AGENT_MODELS, CODE_TASKS, 2)
for model in AGENT_MODELS:
    original = np.mean([r["passed"] for r in code_records if r["model"] == model])
    rewritten = np.mean([r["passed"] for r in perturbed_records if r["model"] == model])
    print(f"{model:<11} pass@1 original {original:.0%} -> perturbed {rewritten:.0%}")

Neither model should lose accuracy under perturbation, and `gpt-5-nano` may even improve,
because the rewritten docstrings state the requirements more explicitly than the
terse originals. This is the caveat of perturbation tests: a rewrite changes difficulty as well as
surface form, so a drop is evidence of memorisation but an unchanged score is only weak evidence
against it.

In [ ]:
def docstring_halves(prompt: str) -> tuple[str, str] | None:
    """Split the last docstring into (prefix up to its midpoint word, rest)."""
    match = list(re.finditer(r'"""(.*?)"""', prompt, re.S))
    if not match:
        return None
    body = match[-1].group(1)
    words = body.split(" ")
    cut = len(" ".join(words[: len(words) // 2]))
    return prompt[: match[-1].start(1) + cut], body[cut:].strip()


def verbatim_score(model: str, prompt: str) -> float | None:
    """Word-sequence similarity (difflib ratio) of the continuation to the true rest."""
    halves = docstring_halves(prompt)
    if halves is None:
        return None
    prefix, rest = halves
    reply = (
        call(
            model,
            [
                {
                    "role": "user",
                    "content": "Continue this text exactly from where it stops, reproducing the original if you know it. "
                    f"Output only the continuation.\n\n{prefix}",
                }
            ],
        ).content
        or ""
    )
    truth, guess = rest.split()[:40], reply.split()[:40]
    return difflib.SequenceMatcher(None, truth, guess).ratio()


for model in AGENT_MODELS:
    with ThreadPoolExecutor(max_workers=16) as pool:
        orig = [
            s
            for s in pool.map(
                lambda i: verbatim_score(model, humaneval[i]["prompt"]), CODE_TASKS
            )
            if s is not None
        ]
        ctrl = [
            s
            for s in pool.map(
                lambda i: verbatim_score(model, perturbed[i]["prompt"]), CODE_TASKS
            )
            if s is not None
        ]
    print(
        f"{model:<11} word-sequence similarity: original docstrings {np.mean(orig):.0%}, "
        f"paraphrased (unseen) {np.mean(ctrl):.0%}"
    )

The verbatim probe shows no memorisation signal either (original and unseen docstrings are
continued equally well, about 15% similarity). Chat models are also trained not to regurgitate,
so this probe is most informative on base models through raw completions or log-probabilities
(Min-K% Prob).

---
# 5) Report card

In [ ]:
PRICES = {
    "gpt-5-mini": (0.25, 2.0),
    "gpt-5-nano": (0.05, 0.4),
    "gpt-4.1-mini": (0.4, 1.6),
}
for name, records, key in [
    ("HumanEval+ agent", code_records, "passed"),
    ("tau-bench retail", tau_records, "reward"),
]:
    print(name)
    for model in AGENT_MODELS:
        mine = [r for r in records if r["model"] == model]
        by_task = group_by_task(records, model, key)
        p1 = np.mean([pass_at_k(len(v), sum(v), 1) for v in by_task.values()])
        pk = np.mean([pass_hat_k(len(v), sum(v), NUM_TRIALS) for v in by_task.values()])
        lo, hi = bootstrap_ci([pass_at_k(len(v), sum(v), 1) for v in by_task.values()])
        print(
            f"  {model:<11} pass@1 {p1:.2f} [{lo:.2f}, {hi:.2f}]  pass^{NUM_TRIALS} {pk:.2f}  "
            f"steps {np.mean([r['steps'] for r in mine]):.1f}"
        )
print("\nTokens and list-price cost of this whole notebook run:")
for model, u in USAGE.items():
    price_in, price_out = PRICES.get(model, (0, 0))
    cost = u["input"] / 1e6 * price_in + u["output"] / 1e6 * price_out
    print(
        f"  {model:<13} {u['calls']:>5} calls, {u['input'] / 1e6:6.2f}M in, {u['output'] / 1e6:5.2f}M out, ~${cost:.2f}"
    )

Prices are list prices per million tokens at the time of writing and change often; the token
counts are what was actually used.

---
# 6) Practical notes

- **Report pass^k, not just pass@1,** for any agent that faces users. It is cheap: the same $n$
  trials give every $k \le n$.
- **The harness is part of the score.** Tool design, step limits, prompts and retry policy move
  SWE-bench Verified results by many points for the same model. Compare models inside one
  harness; compare harnesses with one model.
- **Grade outcomes in the environment.** Hidden tests and database state are robust to phrasing.
  Judges are for process qualities that have no programmatic check, and need validation against
  outcomes before use.
- **User simulators are a source of variance.** τ²-bench and later versions constrain the
  simulated user with tools and stricter instructions; a sloppy simulator makes good agents look
  bad.
- **Contamination defences.** Time-split benchmarks (LiveCodeBench, SWE-bench-Live), private
  held-out sets, and perturbation studies. A large drop under perturbation is a red flag; no drop
  is weak evidence of no contamination.
- **Cost and latency are metrics.** Report tokens and wall-clock per task next to accuracy; an
  agent that solves 2% more tasks at 3× the cost is often the wrong choice.
- **Sandboxing.** Model-written code needs isolation from the host (Seatbelt, Landlock, gVisor,
  Firecracker microVMs), no default network, and resource limits.

---
# 7) Conclusion

| Built | Takeaway |
|---|---|
| pass@k and pass^k estimators | Same trials, two questions: can it ever succeed vs does it reliably succeed |
| Bootstrap intervals over tasks | Tens of tasks give wide intervals; differences inside them are noise |
| Sandboxed code harness with hidden tests | Terminal-Bench / SWE-bench structure: tools, workspace, grading after submit |
| τ-bench retail harness | Real database, tools, policy and tasks; simulated user; database-state reward |
| Trajectory judge | Useful for process quality, validated against outcome reward before trusting it |
| Contamination probes | Perturbation and verbatim-completion tests need no training-data access |